# 🧠 Neuriq: Stacking XGForest EEG Training Pipeline
### Advanced Multi-Class Brainwave Analytics for Psychiatric Disorder Detection
---
This Jupyter Notebook implements the exact same backend model training and preprocessing pipeline used in the **Neuriq AI** FastAPI backend. 

### 🔬 Stacking Ensemble Pipeline
1. **Dataset Generation / Ingestion (Step 1)**: Ingests 1,200 patient records with 95 multi-channel spectral power columns representing Delta, Theta, Alpha, Beta, and Gamma bands across 19 standard channels (10-20 EEG Electrode System).
2. **Clinical Preprocessing & Artifact Filtering (Step 2)**: Reconstructs missing features (imputation), drops duplicates, clamps massive muscle outlier spikes ($Z \ge 3.0$), and maps spectral features to a scale-free $[0, 1]$ range.
3. **PCA-RFE Hybrid Feature Selection (Step 3)**: Performs Principal Component Analysis (PCA) to extract $95$ orthogonal components, then applies Recursive Feature Elimination (RFE) using a Random Forest estimator to select the top $10$ predictive features.
4. **GridSearchCV Stacking Assembly (Step 4 & 5)**: Uses multi-threaded grid search with 5-fold cross-validation to lock in parameters for Random Forest and XGBoost base estimators, then trains a Logistic Regression meta-model.
5. **Clinical Visualization & Serialization**: Plots confusion matrix, ROC-AUC curves, band importances, and PCA explained variance. Serializes pipeline weights directly into the backend checkpoint directories (`ml_service/models/checkpoints/`).

In [ ]:
# ── ENVIRONMENT SETUP & WORKING DIRECTORY ALIGNMENT ─────────────────────────
import os
import sys

# Align current working directory to workspace root to ensure path compatibility
cwd = os.getcwd()
print(f"Initial CWD: {cwd}")
if cwd.endswith("notebooks") or cwd.endswith("ml_service"):
    os.chdir("..")
print(f"Aligned Working Directory: {os.getcwd()}")

# Verify ml_service is in python search path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Ensure output directories exist
os.makedirs("ml_service/models/checkpoints", exist_ok=True)
os.makedirs("ml_service/results", exist_ok=True)
os.makedirs("ml_service/datasets", exist_ok=True)

## 📦 Step 0: Dependencies & Imports
Let's import all scientific, mathematical, machine learning, and visualization libraries. We will also set our plot styles to get premium, publication-grade aesthetics.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict, Any

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_fscore_support
from sklearn.preprocessing import LabelBinarizer
import xgboost as xgb

# Set style for premium visualizations
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('ggplot')
    
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# Custom CSS-like print coloring for clean output
class TermColors:
    HEADER = '\033[95m'
    BLUE = '\033[94m'
    GREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    END = '\033[0m'
    BOLD = '\033[1m'

print(f"{TermColors.GREEN}✓ Environment successfully configured and dependencies loaded!{TermColors.END}")

## 📊 Step 1: Multi-Class EEG Dataset Generation
We will generate/load the highly rigorous EEG dataset representing the **Kaggle Multi-Class EEG Dataset for Psychiatric Disorders**.

It generates **1,200 patient recordings** across 3 categories:
- **Class 0 (Healthy Controls)**: Rear posterior alpha rhythm dominance.
- **Class 1 (Social Anxiety Disorder)**: Frontal beta/gamma hyperactivation, frontal alpha hypoactivity.
- **Class 2 (Acute Stress State)**: Widespread high beta, blocked alpha, elevated delta/theta slow waves.

The generator also injects missing values, duplicate records, and massive muscle/outlier voltage spikes to challenge the preprocessing pipeline.

In [ ]:
# Programmatic EEG Generator
N_PATIENTS = 1200
CHANNELS = [
    'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz'
]
BANDS = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']
feature_names = [f"PSD_{band}_{channel}" for channel in CHANNELS for band in BANDS]

def local_generate_dataset() -> pd.DataFrame:
    np.random.seed(42)
    records = []
    labels = np.repeat([0, 1, 2], N_PATIENTS // 3)
    
    for idx, label in enumerate(labels):
        record = {
            "patient_id": f"SUBJ_{idx:04d}",
            "age": max(18, min(70, int(np.random.normal(32.0 if label == 0 else (28.0 if label == 1 else 35.0), 8.0)))),
            "sex": np.random.choice(["M", "F"], p=[0.48, 0.52])
        }
        
        for ch in CHANNELS:
            is_frontal = ch in ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz']
            is_posterior = ch in ['P3', 'P4', 'O1', 'O2', 'Pz']
            
            for band in BANDS:
                col_name = f"PSD_{band}_{ch}"
                mu, sigma = 15.0, 3.5
                
                if label == 0:  # Healthy Controls
                    if band == 'Alpha' and is_posterior:
                        mu, sigma = 28.0, 4.0
                    elif band in ['Beta', 'Gamma']:
                        mu, sigma = 10.0, 2.0
                elif label == 1:  # Social Anxiety
                    if band in ['Beta', 'Gamma'] and is_frontal:
                        mu, sigma = 29.5, 5.0
                    elif band == 'Alpha' and is_frontal:
                        mu, sigma = 8.5, 1.5
                    elif band == 'Theta':
                        mu, sigma = 22.0, 3.5
                elif label == 2:  # Acute Stress
                    if band == 'Beta':
                        mu, sigma = 32.0, 5.5
                    elif band == 'Alpha':
                        mu, sigma = 9.0, 2.0
                    elif band == 'Delta':
                        mu, sigma = 24.0, 4.0
                
                val = np.random.normal(mu, sigma)
                record[col_name] = round(max(0.1, val), 4)
                
        record["label"] = label
        records.append(record)
        
    df = pd.DataFrame(records)
    
    # Inject dropouts (NaNs)
    mask_missing = np.random.random(size=(N_PATIENTS, len(feature_names))) < 0.03
    for col_idx, col_name in enumerate(feature_names):
        df.loc[mask_missing[:, col_idx], col_name] = np.nan
        
    # Inject outlier spikes
    for _ in range(50):
        r = np.random.randint(0, N_PATIENTS)
        c = np.random.choice(feature_names)
        df.loc[r, c] = np.random.choice([180.0, 220.0])
        
    # Inject duplicate rows
    dup_rows = df.sample(n=15, random_state=42)
    df = pd.concat([df, dup_rows], ignore_index=True)
    return df

data_path = "ml_service/datasets/eeg_mental_health.csv"
if not os.path.exists(data_path):
    print(f"{TermColors.WARNING}Dataset not found at {data_path}. Running local generator...{TermColors.END}")
    df = local_generate_dataset()
    df.to_csv(data_path, index=False)
    print(f"{TermColors.GREEN}✓ Successfully generated and saved dataset! Shape: {df.shape}{TermColors.END}")
else:
    df = pd.read_csv(data_path)
    print(f"{TermColors.GREEN}✓ Loaded existing EEG dataset with shape: {df.shape}{TermColors.END}")

# Quick inspect of cohort counts
print("Cohort distribution:")
print(df['label'].value_counts())

## 🔬 Step 2: Clinical Preprocessing & Signal Cleaning
Real brainwave signals are notoriously noisy, corrupted by muscle activity, sweat, ocular blinks, and machine impedance. 
Our pipeline performs **four strict preprocessing subroutines**:
1. **Statistical Imputation**: Identifies NaNs and replaces them using calculated column average values.
2. **Duplicate Removal**: Removes duplicate rows to safeguard against testing/training bias.
3. **Outlier Z-score Clamping**: Evaluates standard scores ($Z$) and clips spikes exceeding $3.0\sigma$ to the boundaries to eliminate muscle artifact voltage spikes.
4. **Min-Max Normalization**: Maps all raw voltages directly into a strict, scale-free $[0, 1]$ range.

In [ ]:
class EEGPreprocessor:
    """Preprocesses highly dimensional brainwave frequency bands according to clinical standards."""
    def __init__(self, z_threshold: float = 3.0):
        self.z_threshold = z_threshold
        self.column_means_ = {}
        self.feature_min_ = {}
        self.feature_max_ = {}
        self.column_stds_ = {}
        
    def fit(self, df: pd.DataFrame, feature_cols: List[str]) -> 'EEGPreprocessor':
        df_clean = df.drop_duplicates()
        for col in feature_cols:
            vals = df_clean[col].values
            mean_val = float(np.nanmean(vals)) if not np.all(np.isnan(vals)) else 0.0
            std_val = float(np.nanstd(vals)) if not np.all(np.isnan(vals)) else 1.0
            std_val = std_val if std_val > 1e-8 else 1.0
            
            imputed = np.where(np.isnan(vals), mean_val, vals)
            z = (imputed - mean_val) / std_val
            clipped = np.where(np.abs(z) > self.z_threshold, mean_val + np.sign(z) * self.z_threshold * std_val, imputed)
            
            self.column_means_[col] = mean_val
            self.column_stds_[col] = std_val
            self.feature_min_[col] = float(np.min(clipped))
            self.feature_max_[col] = float(np.max(clipped))
        return self
        
    def transform(self, df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        df_out = df.copy()
        # 1. Imputation
        for col in feature_cols:
            mean_val = self.column_means_.get(col, 0.0)
            df_out[col] = df_out[col].fillna(mean_val)
            
        # 2. Outlier Clamping (Z-score)
        for col in feature_cols:
            mean_val = self.column_means_.get(col, 0.0)
            std_val = self.column_stds_.get(col, 1.0)
            vals = df_out[col].values
            z_scores = (vals - mean_val) / std_val
            outliers = np.abs(z_scores) > self.z_threshold
            vals[outliers] = mean_val + np.sign(z_scores[outliers]) * self.z_threshold * std_val
            df_out[col] = vals
            
        # 3. Min-Max Scaling
        for col in feature_cols:
            min_val = self.feature_min_.get(col, 0.0)
            max_val = self.feature_max_.get(col, 1.0)
            diff = max_val - min_val
            if diff < 1e-8:
                df_out[col] = 0.0
            else:
                df_out[col] = (df_out[col] - min_val) / diff
                df_out[col] = np.clip(df_out[col], 0.0, 1.0)
        return df_out

    def clean_and_preprocess_dataset(self, df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        df_cleaned = df.drop_duplicates()
        self.fit(df_cleaned, feature_cols)
        return self.transform(df_cleaned, feature_cols)

# Run preprocessing
print("Applying preprocessor pipeline...")
preprocessor = EEGPreprocessor(z_threshold=3.0)
df_clean = preprocessor.clean_and_preprocess_dataset(df, feature_names)

print(f"Raw dataset duplicate count: {len(df) - len(df_clean)}")
print(f"Final preprocessed features shape: {df_clean[feature_names].shape}")
print(f"NaN counts in preprocessed data: {df_clean[feature_names].isna().sum().sum()}")
print(f"Min value: {df_clean[feature_names].values.min():.2f} | Max value: {df_clean[feature_names].values.max():.2f}")

### 🔬 Preprocessing Visual Validation
Let's visual-verify how the outlier clamping affects high voltage artifact spikes. 
We will plot the original raw values vs. the preprocessed and scaled values of the frontal electrode.

In [ ]:
# Let's visualize raw vs. cleaned alpha features
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["PSD_Alpha_Fp1"], kde=True, ax=ax[0], color="#EF5350")
ax[0].set_title("Raw Alpha PSD (with Outlier Artifact Spikes)", fontsize=12, fontweight="bold")
ax[0].set_xlabel("PSD (uV²)")

sns.histplot(df_clean["PSD_Alpha_Fp1"], kde=True, ax=ax[1], color="#00FFCC")
ax[1].set_title("Preprocessed & Scale-Free [0, 1] Alpha PSD", fontsize=12, fontweight="bold")
ax[1].set_xlabel("Scaled Value")

plt.tight_layout()
plt.show()

## 🧬 Step 3: PCA-RFE Hybrid Feature Selection
Having 95 continuous features poses high collinearity and dimensionality risks.
Our hybrid pipeline resolves this through a **two-phase feature extraction engine**:
- **Phase 1: Principal Component Analysis (PCA)**: Projects the highly correlated 95 channels into a completely orthogonal principal component space representing maximal dataset variance.
- **Phase 2: Recursive Feature Elimination (RFE)**: Fits a forest tree classifier directly in the principal component coordinate system and recursively eliminates low-utility components until only the **top 10 principal components** remain.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE

class PCARFEHybridSelector:
    """Combines Principal Component Analysis and Recursive Feature Elimination."""
    def __init__(self, pca_components: int = 95, rfe_select_components: int = 10, random_state: int = 42):
        self.pca_components = pca_components
        self.rfe_select_components = rfe_select_components
        self.random_state = random_state
        
        self.pca = PCA(n_components=self.pca_components, random_state=self.random_state)
        self.estimator = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=self.random_state)
        self.rfe = RFE(estimator=self.estimator, n_features_to_select=self.rfe_select_components, step=1)
        
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'PCARFEHybridSelector':
        X = np.nan_to_num(X)
        n_comp = min(self.pca_components, X.shape[1])
        if n_comp != self.pca.n_components:
            self.pca = PCA(n_components=n_comp, random_state=self.random_state)
            
        X_pca = self.pca.fit_transform(X)
        n_sel = min(self.rfe_select_components, X_pca.shape[1])
        if n_sel != self.rfe.n_features_to_select:
            self.rfe = RFE(estimator=self.estimator, n_features_to_select=n_sel, step=1)
            
        self.rfe.fit(X_pca, y)
        return self
        
    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.nan_to_num(X)
        X_pca = self.pca.transform(X)
        return self.rfe.transform(X_pca)
        
    def fit_transform(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        return self.fit(X, y).transform(X)
        
    def get_explained_variance_ratio(self) -> np.ndarray:
        return self.pca.explained_variance_ratio_
        
    def get_selected_components_indices(self) -> List[int]:
        support = self.rfe.support_
        return [i for i, val in enumerate(support) if val]

# Split data
X = df_clean[feature_names].values
y = df_clean["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Fitting PCA-RFE Hybrid Feature Selector...")
selector = PCARFEHybridSelector(pca_components=95, rfe_select_components=10)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_components = selector.get_selected_components_indices()
print(f"{TermColors.GREEN}✓ Feature extraction completed successfully!{TermColors.END}")
print(f"Selected PCA component indices: {selected_components}")
print(f"Shape of training input after PCA-RFE: {X_train_selected.shape}")

In [ ]:
# Let's visualize Explained Variance Ratio of individual Principal Components
var_ratio = selector.get_explained_variance_ratio()
cum_var = np.cumsum(var_ratio) * 100

fig, ax = plt.subplots(figsize=(10, 5))
components_labels = [f"PC {i+1}" for i in range(15)]
sns.barplot(x=components_labels, y=var_ratio[:15] * 100, color="#7000FF", alpha=0.8, ax=ax)
ax.plot(range(15), cum_var[:15], color="#00FFCC", marker='o', linewidth=2, label="Cumulative Explained Variance (%)")
ax.set_title("PCA Explained Variance Ratio & Cumulative Curve", fontsize=13, fontweight="bold")
ax.set_ylabel("Percentage of Explained Variance (%)")
ax.set_xlabel("Principal Components")
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

## 🤖 Step 4: Stacking Ensemble Architecture & GridSearchCV Optimization
To model non-linear boundaries in the low-dimensional PCA-RFE latent space, we build the **XGForest Stacking Ensemble**:
1. **Base Estimator A: Random Forest Classifier**: Highly robust tree ensemble specialized in high-variance prevention.
2. **Base Estimator B: XGBoost Classifier**: Optimized gradient boosting trees specialized in low-bias tuning.
3. **Meta-Estimator: Logistic Regression**: Integrates soft predicted class probabilities from both base learners to make the final psychiatric diagnostic prediction.

We run **GridSearchCV sweeps** with 5-fold cross-validation on both base models first to secure hyperparameter alignment before stacking.

In [ ]:
# 1. Optimize Random Forest Base Model
print("Running GridSearchCV for Random Forest (5-fold CV)...")
rf_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 8],
    "min_samples_split": [2, 5]
}
rf_base = RandomForestClassifier(random_state=42)
rf_search = GridSearchCV(estimator=rf_base, param_grid=rf_grid, cv=5, scoring="accuracy", n_jobs=-1)
rf_search.fit(X_train_selected, y_train)
best_rf = rf_search.best_estimator_
print(f"{TermColors.BLUE}Random Forest Best Hyperparameters: {rf_search.best_params_}{TermColors.END}")

# 2. Optimize XGBoost Base Model
print("Running GridSearchCV for XGBoost (5-fold CV)...")
xgb_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 5]
}
xgb_base = xgb.XGBClassifier(objective="multi:softprob", num_class=3, random_state=42, eval_metric="mlogloss")
xgb_search = GridSearchCV(estimator=xgb_base, param_grid=xgb_grid, cv=5, scoring="accuracy", n_jobs=-1)
xgb_search.fit(X_train_selected, y_train)
best_xgb = xgb_search.best_estimator_
print(f"{TermColors.BLUE}XGBoost Best Hyperparameters: {xgb_search.best_params_}{TermColors.END}")

# 3. Assemble XGForest Stacking Ensemble
print("Assembling and training XGForest Stacking Ensemble...")
base_estimators = [
    ("rf", best_rf),
    ("xgb", best_xgb)
]
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    n_jobs=-1
)
stacking_clf.fit(X_train_selected, y_train)
print(f"{TermColors.GREEN}✓ Stacking Classifier fitted and locked!{TermColors.END}")

## 📈 Step 5: Clinical Rigor & Validation Metrics
Let's perform inference over the test cohort, export the detailed classification metrics report, and create **premium visualizations** illustrating performance diagnostics:

In [ ]:
# Inference
y_pred = stacking_clf.predict(X_test_selected)
y_proba = stacking_clf.predict_proba(X_test_selected)

accuracy = float(np.mean(y_pred == y_test))
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

print(f"{TermColors.BOLD}--- Stacking Classifier Clinical Milestones ---{TermColors.END}")
print(f"Test Set Accuracy : {accuracy:.4f}")
print(f"Macro Precision   : {precision:.4f}")
print(f"Macro Recall      : {recall:.4f}")
print(f"Macro F1-Score    : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Healthy Controls", "Social Anxiety", "Acute Stress"]))

In [ ]:
# Draw beautiful dashboard visual components 
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="viridis", cbar=False,
            xticklabels=["Healthy", "Anxiety", "Stress"], 
            yticklabels=["Healthy", "Anxiety", "Stress"], ax=ax[0])
ax[0].set_title("Diagnostic Confusion Matrix Heatmap", fontsize=13, fontweight="bold")
ax[0].set_xlabel("Predicted State")
ax[0].set_ylabel("True State")

# 2. Multi-Class ROC Curves
lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test)
colors = ["#00E676", "#FFA726", "#EF5350"]
labels = ["Healthy Controls", "Social Anxiety", "Acute Stress"]
roc_data = {}

for cl in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, cl], y_proba[:, cl])
    roc_auc = auc(fpr, tpr)
    roc_data[str(cl)] = {
        "fpr": fpr.tolist(),
        "tpr": tpr.tolist(),
        "auc": float(roc_auc)
    }
    ax[1].plot(fpr, tpr, color=colors[cl], label=f"{labels[cl]} (AUC={roc_auc:.2f})", linewidth=2.5)

ax[1].plot([0, 1], [0, 1], 'k--', color="gray", label="Chance Level")
ax[1].set_title("Multi-Class ROC Curves (FPR vs TPR)", fontsize=13, fontweight="bold")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

### 🔬 Biological Rhythms Explainability
Let's identify which underlying brain rhythms dominate the predictions. We fit an auxiliary Random Forest on the entire dataset to compute standard feature importances, then aggregate them by standard bands (Delta, Theta, Alpha, Beta, Gamma).

In [ ]:
aux_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
aux_rf.fit(X_train, y_train)
importances = aux_rf.feature_importances_

band_importances = {"Delta": 0.0, "Theta": 0.0, "Alpha": 0.0, "Beta": 0.0, "Gamma": 0.0}
for col_name, imp in zip(feature_names, importances):
    for band in band_importances.keys():
        if f"_{band}_" in col_name:
            band_importances[band] += float(imp)
            break
            
total_imp = sum(band_importances.values())
if total_imp > 0:
    band_importances = {k: v / total_imp for k, v in band_importances.items()}

# Bar plot
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=list(band_importances.keys()), y=list(band_importances.values()), palette="viridis", ax=ax)
ax.set_title("EEG Feature Importance Weight by Brainwave Rhythm", fontsize=13, fontweight="bold")
ax.set_ylabel("Aggregated Importance Weight")
ax.set_xlabel("EEG Frequency Bands")
plt.tight_layout()
plt.show()

## 💾 Step 6: Pipeline Model Checkpoint Serializations
To make our newly trained notebook model available to the FastAPI backend API and the stream-lit Analytics Dashboard, we serialize the pipeline components and save the metrics JSON exactly to the backend directories.

In [ ]:
print("Saving model weights checkpoints...")
joblib.dump(stacking_clf, "ml_service/models/checkpoints/best_stacking_model.joblib")
joblib.dump(preprocessor, "ml_service/models/checkpoints/eeg_preprocessor.joblib")
joblib.dump(selector, "ml_service/models/checkpoints/eeg_selector.joblib")

# Assemble and export standard metrics evaluation JSON
metrics_summary = {
    "accuracy": round(accuracy, 4),
    "precision": round(float(precision), 4),
    "recall": round(float(recall), 4),
    "f1_score": round(float(f1), 4),
    "confusion_matrix": cm.tolist(),
    "roc_auc": roc_data,
    "selected_components": selected_components,
    "band_importances": band_importances,
    "explained_variance_ratio": selector.get_explained_variance_ratio().tolist()[:10],
    "hyperparameters": {
        "rf": rf_search.best_params_,
        "xgb": xgb_search.best_params_
    }
}

with open("ml_service/results/eeg_model_evaluation.json", "w") as f:
    json.dump(metrics_summary, f, indent=4)
    
print(f"{TermColors.GREEN}✓ Stacking classifier models and metrics serialized successfully!{TermColors.END}")
print(f"{TermColors.BOLD}✓ Notebook training run fully completed!{TermColors.END}")